In [1]:
from reasonable_crowd.dataset import load_annotations, build_evaluation_dataset
import os
import numpy as np
path_to_reasonable_crowd = "../../Reasonable-Crowd"
path_to_workers = os.path.join(path_to_reasonable_crowd, "annotations/workers.txt")

with open(path_to_workers, "r") as f:
    workers = f.readlines()


annotations = load_annotations(path_to_reasonable_crowd)
X, y, y_votes, y_agreement = build_evaluation_dataset(annotations)

/opt/anaconda3/envs/publish/lib/python3.8/site-packages/scenic/core/errors.py:271: UserWarning: unable to install sys.excepthook to format Scenic backtraces
  warnings.warn("unable to install sys.excepthook to format Scenic backtraces")


In [2]:
num_pairs = 0
for scenario, pairs in annotations.items():
    num_pairs += len(pairs)
print("Number of pairs:", num_pairs)

Number of pairs: 3364


In [3]:
worker_preferences = {}
for worker in workers:
    w_id = worker.strip()
    worker_preferences[w_id] = set()
    
for scenario, pairs in annotations.items():
    for pair_id, votes in pairs.items():
        for w_id, preferences in worker_preferences.items():
            if w_id in votes:
                t1, t2 = pair_id.split(" ;; " )
                preferences.add((t1, t2))
                

In [4]:
agreements = []
for w_id, preferences in worker_preferences.items():
    pairs = set()
    for pref in preferences:
        pairs.add(pref)
        pairs.add(pref[::-1])
        
    for other_w_id, other_preferences in worker_preferences.items():
        if w_id == other_w_id:
            continue
        other_pairs = set()
        for pref in other_preferences:
            other_pairs.add(pref)
            other_pairs.add(pref[::-1])
        total_count = len(pairs.intersection(other_pairs))//2
        if total_count == 0:
            continue
        common_preferences = preferences.intersection(other_preferences)
        agreement_count = len(common_preferences)
        agreement = agreement_count / total_count if total_count > 0 else 0
        agreements.append(agreement)
agreements.sort()

agreements = np.array(agreements)

print("Annotator agreement statistics:")
print("Min agreement:", np.min(agreements))
print("Max agreement:", np.max(agreements))
print("Mean agreement:", np.mean(agreements))
print("Median agreement:", np.median(agreements))
print(len(agreements), "annotator pairs compared.")

Annotator agreement statistics:
Min agreement: 0.0
Max agreement: 1.0
Mean agreement: 0.8073593289092988
Median agreement: 0.8235887096774194
3002 annotator pairs compared.


In [5]:
total_shared_tasks = 0
total_matches = 0

for w_id, preferences in worker_preferences.items():
    pairs = set()
    for pref in preferences:
        pairs.add(pref)
        pairs.add(pref[::-1])
    
    for other_w_id, other_preferences in worker_preferences.items():
        if w_id >= other_w_id: # Avoids double-counting A-B and B-A
            continue
            
        overlap_count = len(pairs.intersection(other_pairs)) // 2
        if overlap_count == 0:
            continue
        
        other_pairs = set()
        for pref in other_preferences:
            other_pairs.add(pref)
            other_pairs.add(pref[::-1])
            
        matches = len(preferences.intersection(other_preferences))
        
        total_shared_tasks += overlap_count
        total_matches += matches

global_agreement = total_matches / total_shared_tasks
print(f"Global Pooled Agreement: {global_agreement:.4f}")

Global Pooled Agreement: 0.8462


In [18]:
worker_scores = {}

for w_id, preferences in worker_preferences.items():
    total_matches = 0
    total_overlap = 0
    
    # Pre-calculate the worker's own expanded pairs once
    w_pairs = set()
    for pref in preferences:
        w_pairs.add(pref)
        w_pairs.add(pref[::-1])

    for other_w_id, other_preferences in worker_preferences.items():
        if w_id == other_w_id:
            continue
            
        # Check intersection
        other_pairs = set()
        for pref in other_preferences:
            other_pairs.add(pref)
            other_pairs.add(pref[::-1])
            
        overlap = len(w_pairs.intersection(other_pairs)) // 2
        if overlap == 0:
            continue
            
        matches = len(preferences.intersection(other_preferences))
        
        total_matches += matches
        total_overlap += overlap
    
    if total_overlap > 0:
        worker_scores[w_id] = total_matches / total_overlap

# Statistics on workers
scores = list(worker_scores.values())
print(f"Mean Worker Reliability: {np.mean(scores)}")

Mean Worker Reliability: 0.8224815502621244


Annotator accuracy statistics:
Min accuracy: 0.5
Max accuracy: 1.0
Mean accuracy: 0.8363332215110086
Median accuracy: 0.8429319371727748
65 annotators evaluated.


In [ ]:
from reasonable_crowd.parse_map import parse_map
from reasonable_crowd.dataset import build_evaluation_dataset, get_trajectories, load_annotations
import os
from rulebook_benchmark.realization import VariableHandler
from rulebook_benchmark.rule_functions import RuleEngine
from rulebook_benchmark.rule_functions import (
    f1, f2, f3, f4, f5, f6, f7, f8, f9, f11, f12, f13, f15, f17, f18, Result
)
from rulebook_benchmark.rulebook import Rulebook
from reasonable_crowd.InPlaceRulebook import InPlaceRulebook
import numpy as np
import pandas as pd
from reasonable_crowd.optimization import cache_rule_evaluations, optimize_rulebook_grid_bruteforce_with_validation, simulated_annealing, simulated_annealing_with_validation, number_of_unique_rulebooks, find_scenario_rulebooks, group_rulebook, optimize_rulebook_greedy_by_priority
import pickle
from sklearn.model_selection import train_test_split
from reasonable_crowd.evaluation import evaluate_rulebook_with_cache
from sklearn.model_selection import KFold
from reasonable_crowd.visualization import plot_topological_graph, plot_two_rulebooks_side_by_side

SEED = 50
NUM_RUNS = 10


path_to_reasonable_crowd = "../../Reasonable-Crowd"
map_directory = path_to_reasonable_crowd + '/maps'
trajectory_directory = path_to_reasonable_crowd + '/trajectories'

network_U = parse_map(map_directory, 'U')
network_S = parse_map(map_directory, 'S')

output_directory = '../outputs'
output_file = os.path.join(output_directory, 'results_scenic.txt')

print("Getting trajectories...")

trajectories = get_trajectories(output_directory, trajectory_directory, network_U, network_S)

trajectories_dict = {}
for filename, realization in trajectories:
    trajectories_dict[filename[:-5]] = realization  # remove .json extension

# create pandas dataframe
df = pd.DataFrame(columns=['X', 'y', 'votes', 'agreement'])
df['X'] = X
df['y'] = y
df['votes'] = y_votes
df['agreement'] = y_agreement

print(df.head())

rb = Rulebook(rule_file="../src/reasonable_crowd/reasonable_crowd_rule_functions.py", rulebook_file="../src/reasonable_crowd/reasonable_crowd_5.graph")
rule_id_to_rule = {1: f1, 2: f2, 3: f3, 4: f4, 5: f5, 6: f6, 7: f7, 8: f8, 9: f9, 11: f11, 12: f12, 13: f13, 15: f15, 17: f17, 18: f18}
rulebook = InPlaceRulebook(rb.priority_graph, rule_id_to_rule)


rule_id_to_params = {4: ["threshold"], 6: ["threshold"], 8: ["threshold"], 9: ["threshold"], 5: ["velocity", "threshold", "timesteps"], 11: ["threshold"], 12: ["threshold"], 13: ["threshold"], 18: ["buffer"]}
rule_id_to_values = {4: {"threshold": [0.6, 0.8, 1, 1.2]}, 6: {"threshold": [0.6, 0.8, 1, 1.2]}, 8: {"threshold": [0.5, 1, 1.5, 2]}, 9: {"threshold": [0.5 , 1, 1.5, 2]}, 5: {"velocity": [3, 4, 5], "threshold": [-1.5, -1, -0.5], "timesteps": [20, 30, 40]}, 11: {"threshold": [0.4, 0.8, 1.2, 1.6]}, 12: {"threshold": [0.4, 0.8, 1.2, 1.6]}, 13: {"threshold": [0.4, 0.8, 1.2, 1.6]}, 18: {"buffer": [0.3, 0.5, 0.7]}}

if os.path.exists(os.path.join(output_directory, 'tuning_cache.pkl')):    
    print("Loading cached rule evaluations...")
    with open(os.path.join(output_directory, 'tuning_cache.pkl'), 'rb') as f:
        cache_dict = pickle.load(f)
else:
    print("No cached rule evaluations found. Starting with empty cache.")
    print("Saving default rulebook parameters...")
    default_params = {}
    for rule_id, rule in rule_id_to_rule.items():
        default_params[rule_id] = rule.parameters.copy()
        print(f"Rule {rule_id} default parameters: {rule.parameters}")
    
    cache_dict = {}
    cache_rule_evaluations(rulebook, rule_id_to_params, rule_id_to_values, X, y, cache_dict, trajectories_dict)
    pickle.dump(cache_dict, open(os.path.join(output_directory, 'tuning_cache.pkl'), 'wb'))
    
    print("Restoring default rulebook parameters...")
    for rule_id, params in default_params.items():
        rule_id_to_rule[rule_id].parameters.update(params)
        

groups = [[1, 2], [3, 7], [8, 9, 11, 12, 13], [17, 18, 15], [4, 5, 6]]
name_to_group = {"safety-critical": groups[0], "operation-limit": groups[1], "safety-enhancing": groups[2], "predictability": groups[3], "precautionary": groups[4]}
group_to_name = {tuple(value): key for key, value in name_to_group.items()}
rulebook = group_rulebook(rulebook, groups, keep_relations=True)


base_result = evaluate_rulebook_with_cache(
    rulebook,
    X,
    y,
    y_votes,
    cache_dict,
    trajectories_dict)


print()
print("Base Rulebook Results:")
print("----------------------")
print("Correct:", base_result[0])
print("Equal:", base_result[1])
print("Incomparable:", base_result[2])
print("Total:", base_result[3])
print("Accuracy:", base_result[4])
print("Weighted Accuracy:", base_result[5])
print("Accuracy out of predictions:", base_result[0]/(base_result[3]-base_result[2]) if base_result[3]-base_result[2]>0 else 0.0)
print("\n")


Getting trajectories...
Loading cached trajectories...
Loading annotations...
Building evaluation dataset...
                X                y    votes  agreement
0  (U_1-a, U_1-b)  Relation.LARGER  (14, 0)   1.000000
1  (U_1-a, U_1-c)  Relation.LARGER  (14, 0)   1.000000
2  (U_1-a, U_1-d)  Relation.LARGER  (12, 2)   0.714286
3  (U_1-a, U_1-e)  Relation.LARGER  (10, 4)   0.428571
4  (U_1-a, U_1-f)  Relation.LARGER  (14, 0)   1.000000
Loading cached rule evaluations...

Base Rulebook Results:
----------------------
Correct: 1361
Equal: 7
Incomparable: 12
Total: 1682
Accuracy: 0.8091557669441142
Weighted Accuracy: 0.8245508622243586
Accuracy out of predictions: 0.8149700598802395




In [23]:
from rulebook_benchmark.rulebook import Relation

y_hat = base_result[-1]
accuracies = []
rulebook_agreements = []

for w_id, preferences in worker_preferences.items():
    correct = 0
    total = 0
    agree = 0
    for pair, label, pred in zip(X, y, y_hat):
        t1, t2 = pair
        if (t1, t2) in preferences:
            decision = Relation.LARGER
        elif (t2, t1) in preferences:
            decision = Relation.SMALLER
        else:
            continue
    
        total += 1
        if decision == label:
            correct += 1
        if decision == pred:
            agree += 1
        
    accuracy = correct / total if total > 0 else 0
    accuracies.append(accuracy)
    rulebook_agreements.append(agree / total if total > 0 else 0)
    
accuracies = np.array(accuracies)
accuracies.sort()

print("Annotator accuracy statistics:")
print("Min accuracy:", np.min(accuracies))
print("Max accuracy:", np.max(accuracies))
print("Mean accuracy:", np.mean(accuracies))
print("Median accuracy:", np.median(accuracies))
print(len(accuracies), "annotators evaluated.")

print("Annotator-rulebook agreement statistics:")
print("Min agreement:", np.min(rulebook_agreements))
print("Max agreement:", np.max(rulebook_agreements))
print("Mean agreement:", np.mean(rulebook_agreements))
print("Median agreement:", np.median(rulebook_agreements))

Annotator accuracy statistics:
Min accuracy: 0.5
Max accuracy: 1.0
Mean accuracy: 0.8363332215110086
Median accuracy: 0.8429319371727748
65 annotators evaluated.
Annotator-rulebook agreement statistics:
Min agreement: 0.375
Max agreement: 1.0
Mean agreement: 0.751891954829579
Median agreement: 0.7417218543046358
